# Flight Delay Classification — Training, Evaluation & Hyperparameter Tuning

This notebook assumes the input dataset has already been **preprocessed and feature engineered**.
Pipeline: load → split → train multiple models → evaluate → tune the best one(s) → final report.

**Models covered:** Logistic Regression, Naive Bayes, Decision Tree, KNN, Random Forest, XGBoost, MLP.

> Adjust `DATA_PATH`, `TARGET_COL`, and `SAMPLE_FRAC` at the top of the *Setup* section to match your data.


## 1. Setup

In [ ]:
# Core
import os, time, json, warnings, joblib
import numpy as np
import pandas as pd
from pathlib import Path

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# XGBoost (optional)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping. Install with: pip install xgboost")

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---- CONFIG: edit these ----
DATA_PATH   = "data/processed_flights.parquet"   # or .csv
TARGET_COL  = "delay_class"                      # name of your label column
SAMPLE_FRAC = 1.0                                # use e.g. 0.1 for fast iteration on 11M rows
OUTPUT_DIR  = Path("artifacts"); OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Load the preprocessed dataset

Parquet is strongly recommended over CSV at this scale — it's ~5–10× faster to load and preserves dtypes.

In [ ]:
def load_data(path, sample_frac=1.0):
    path = Path(path)
    if path.suffix == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix in (".csv", ".tsv"):
        sep = "," if path.suffix == ".csv" else "\t"
        df = pd.read_csv(path, sep=sep)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    if sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=RANDOM_STATE).reset_index(drop=True)
    return df

df = load_data(DATA_PATH, SAMPLE_FRAC)
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()


In [ ]:
# Quick sanity checks
print("Missing values per column (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))
print("\nTarget distribution:")
print(df[TARGET_COL].value_counts(normalize=True).round(4))


## 3. Train / Validation / Test split

For a time-series dataset you have two reasonable options:

1. **Random stratified split** — use this if your features already encode temporal information (month, day-of-week, hour, etc.) and rows are treated as independent samples. Common choice for flight-delay classification.
2. **Chronological split** — use this if you want to *forecast* future delays from past data. Sort by date, then take the earliest 70% for train, next 15% for validation, last 15% for test.

The cell below does the random stratified split. Swap to the chronological version if needed.

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Random stratified split (default)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765,  # 0.1765 * 0.85 ~= 0.15
    stratify=y_trainval, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape},  Val: {X_val.shape},  Test: {X_test.shape}")

# ---- Alternative: chronological split (uncomment if you have a 'date' column) ----
# df_sorted = df.sort_values("date").reset_index(drop=True)
# n = len(df_sorted)
# train_end = int(0.70 * n); val_end = int(0.85 * n)
# X_train, y_train = df_sorted.iloc[:train_end].drop(columns=[TARGET_COL]), df_sorted.iloc[:train_end][TARGET_COL]
# X_val,   y_val   = df_sorted.iloc[train_end:val_end].drop(columns=[TARGET_COL]), df_sorted.iloc[train_end:val_end][TARGET_COL]
# X_test,  y_test  = df_sorted.iloc[val_end:].drop(columns=[TARGET_COL]), df_sorted.iloc[val_end:][TARGET_COL]


## 4. Define models

Each model is wrapped in a `Pipeline` so scaling is applied only where it's needed (LogReg, KNN, MLP) and skipped for the tree-based ones (which don't need it).

Notes for the dataset's scale (~11M rows):
- **KNN** is set with `n_jobs=-1`; even so it will be slow. Consider sub-sampling for KNN, or swap to FAISS-based approximate search.
- **MLP** with sklearn is CPU-only and slow on big data. For serious work, use PyTorch or Keras.
- **XGBoost** with `tree_method='hist'` is the fastest practical option here.

In [ ]:
def make_models():
    sc = StandardScaler()
    models = {
        "LogisticRegression": Pipeline([
            ("scaler", sc),
            ("clf", LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)),
        ]),
        "NaiveBayes": Pipeline([
            ("clf", GaussianNB()),
        ]),
        "DecisionTree": Pipeline([
            ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]),
        "KNN": Pipeline([
            ("scaler", sc),
            ("clf", KNeighborsClassifier(n_neighbors=15, n_jobs=-1)),
        ]),
        "RandomForest": Pipeline([
            ("clf", RandomForestClassifier(
                n_estimators=200, max_depth=20, n_jobs=-1, random_state=RANDOM_STATE
            )),
        ]),
        "MLP": Pipeline([
            ("scaler", sc),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(64, 32), max_iter=30,
                early_stopping=True, random_state=RANDOM_STATE
            )),
        ]),
    }
    if HAS_XGB:
        models["XGBoost"] = Pipeline([
            ("clf", XGBClassifier(
                n_estimators=300, max_depth=8, learning_rate=0.1,
                tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
                eval_metric="logloss",
            )),
        ])
    return models

models = make_models()
list(models.keys())


## 5. Baseline training & evaluation

Train each model on `X_train`, evaluate on `X_val`. Metrics are collected into a dataframe so models can be compared at a glance.

In [ ]:
def evaluate(y_true, y_pred, y_proba=None, average="weighted"):
    metrics = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall":    recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1":        f1_score(y_true, y_pred, average=average, zero_division=0),
    }
    if y_proba is not None:
        try:
            if y_proba.ndim == 2 and y_proba.shape[1] == 2:
                metrics["roc_auc"] = roc_auc_score(y_true, y_proba[:, 1])
            else:
                metrics["roc_auc"] = roc_auc_score(y_true, y_proba, multi_class="ovr", average=average)
        except Exception:
            metrics["roc_auc"] = np.nan
    return metrics

results = []
trained = {}

for name, pipe in models.items():
    print(f"\n=== Training {name} ===")
    t0 = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = pipe.predict(X_val)
    pred_time = time.time() - t0

    y_proba = pipe.predict_proba(X_val) if hasattr(pipe.named_steps["clf"], "predict_proba") else None
    m = evaluate(y_val, y_pred, y_proba)
    m.update({"model": name, "train_time_s": round(train_time, 2), "pred_time_s": round(pred_time, 2)})
    results.append(m)
    trained[name] = pipe

    print(f"  acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  train={train_time:.1f}s")

results_df = pd.DataFrame(results).set_index("model").sort_values("f1", ascending=False)
results_df


In [ ]:
# Visualize baseline comparison
fig, ax = plt.subplots(figsize=(10, 5))
results_df[["accuracy", "precision", "recall", "f1"]].plot.bar(ax=ax)
ax.set_title("Baseline model comparison (validation set)")
ax.set_ylabel("Score"); ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "baseline_comparison.png", dpi=120)
plt.show()


## 6. Confusion matrices for the top models

In [ ]:
top_n = 3
top_models = results_df.head(top_n).index.tolist()
fig, axes = plt.subplots(1, top_n, figsize=(5 * top_n, 4))
if top_n == 1: axes = [axes]

for ax, name in zip(axes, top_models):
    y_pred = trained[name].predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=sorted(y_val.unique())).plot(ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=120)
plt.show()


## 7. Hyperparameter tuning

`RandomizedSearchCV` is preferred over `GridSearchCV` at this scale — it samples from each distribution
rather than exhaustively trying every combination. Set `n_iter` higher for a more thorough search.

`StratifiedKFold` is used so each fold preserves the class balance. For very large datasets, set `cv=3` to save time.

In [ ]:
# Search spaces — extend or trim to taste
param_distributions = {
    "LogisticRegression": {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs", "saga"],
    },
    "DecisionTree": {
        "clf__max_depth": [5, 10, 20, 30, None],
        "clf__min_samples_split": [2, 10, 50],
        "clf__criterion": ["gini", "entropy"],
    },
    "KNN": {
        "clf__n_neighbors": [5, 15, 25, 50],
        "clf__weights": ["uniform", "distance"],
        "clf__p": [1, 2],   # 1=manhattan, 2=euclidean
    },
    "RandomForest": {
        "clf__n_estimators": [100, 200, 400],
        "clf__max_depth": [10, 20, 30, None],
        "clf__min_samples_split": [2, 10, 50],
        "clf__max_features": ["sqrt", "log2"],
    },
}

if HAS_XGB:
    param_distributions["XGBoost"] = {
        "clf__n_estimators": [200, 400, 600],
        "clf__max_depth": [4, 6, 8, 10],
        "clf__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "clf__subsample": [0.7, 0.85, 1.0],
        "clf__colsample_bytree": [0.7, 0.85, 1.0],
    }


In [ ]:
# Tune the top models from the baseline run
TUNE_TOP_K = 2     # how many of the best baseline models to tune
N_ITER = 15        # number of random parameter combinations to try
CV_FOLDS = 3

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
tuned = {}
tuning_results = []

candidates = [m for m in results_df.head(TUNE_TOP_K).index if m in param_distributions]
print(f"Tuning: {candidates}")

for name in candidates:
    print(f"\n--- Tuning {name} ---")
    search = RandomizedSearchCV(
        estimator=models[name],
        param_distributions=param_distributions[name],
        n_iter=N_ITER,
        scoring="f1_weighted",
        cv=cv,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1,
        refit=True,
    )
    t0 = time.time()
    search.fit(X_train, y_train)
    elapsed = time.time() - t0

    tuned[name] = search.best_estimator_
    print(f"  best CV f1 = {search.best_score_:.4f}  ({elapsed:.1f}s)")
    print(f"  best params = {search.best_params_}")

    tuning_results.append({
        "model": name,
        "best_cv_f1": search.best_score_,
        "best_params": search.best_params_,
        "tune_time_s": round(elapsed, 1),
    })

tuning_df = pd.DataFrame(tuning_results)
tuning_df


## 8. Evaluate tuned models on the held-out test set

Up to this point the test set has never been touched. This is the only honest measure of generalization performance.

In [ ]:
final_results = []

for name, pipe in tuned.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test) if hasattr(pipe.named_steps["clf"], "predict_proba") else None
    m = evaluate(y_test, y_pred, y_proba)
    m["model"] = name
    final_results.append(m)
    print(f"\n=== {name} on TEST ===")
    print(classification_report(y_test, y_pred, zero_division=0))

final_df = pd.DataFrame(final_results).set_index("model").sort_values("f1", ascending=False)
final_df


## 9. Save the best model

In [ ]:
best_name = final_df["f1"].idxmax()
best_model = tuned[best_name]
best_path = OUTPUT_DIR / f"best_model_{best_name}.joblib"
joblib.dump(best_model, best_path)
print(f"Saved {best_name} to {best_path}")

# Persist the metrics tables for the report
results_df.to_csv(OUTPUT_DIR / "baseline_metrics.csv")
final_df.to_csv(OUTPUT_DIR / "tuned_test_metrics.csv")
tuning_df.to_csv(OUTPUT_DIR / "tuning_summary.csv", index=False)
print("Saved metrics CSVs to", OUTPUT_DIR)


## 10. Notes & next steps

- **Class imbalance:** if your delay classes are skewed, set `class_weight="balanced"` on the linear/tree models, or pass `scale_pos_weight` to XGBoost.
- **Feature importance:** `tuned['XGBoost'].named_steps['clf'].feature_importances_` (or use SHAP for proper explanations).
- **Faster experimentation:** keep `SAMPLE_FRAC=0.1` while iterating, then set it back to `1.0` for the final run.
- **LightGBM:** if XGBoost is too slow, swap it for LightGBM — drop-in API, lower memory, faster on >10M rows.
